In [ ]:
using ArgParse
using Printf
using Dates
using JLD2
using ITensors
using ITensorMPS
using LinearAlgebra

const ROOT = normpath(joinpath(@__DIR__, ".."))
include(joinpath(ROOT, "QCSB", "QCSB.jl"))

include(joinpath(ROOT, "src", "circuit.jl"))
include(joinpath(ROOT, "src", "tci.jl"))

tci (generic function with 1 method)

In [136]:
function one_bra(site::Index)
    return sum([dag(ITensors.state(site, "$i")) for i in 0:dim(site)-1])
end

function lognorma(state::DiagonalStateMPS)
    return lognorm(state.mps)
end

function lognorm(ψ::MPS)
    sites = siteinds(ψ)
    L = length(sites)

    mats = Vector{ITensor}(undef, L)
    for j in 1:L
        s = sites[j]
        bra = one_bra(s)
        mats[j] = bra * ψ[j]
    end

    v₀ = mats[end]
    logscale = 0.0
    for j in 1:L-1
        v₁ = mats[end-j] * v₀
        n = norm(v₁)
        logscale += log(n)
        v₀ = v₁ / n
    end

    x = logscale + log(Complex(Array(v₀)[]))
    if abs(imag(x)) < 1e-10
        x = real(x)
    end

    return x
end

function normalize!(state::DiagonalStateMPS)
    logscale = lognorm(state)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites)

    scale = exp(logscale / L)
    for i in 1:L
        state.mps[i] /= scale
    end
    return state
end

function number_left(ψ::MPS)
    sites = siteinds(ψ)
    L = length(sites)

    mats = Vector{ITensor}(undef, L-1)
    for j in 2:L
        s = sites[j]
        bra = one_bra(s)
        mats[j-1] = bra * ψ[j]
    end

    v₀ = mats[end]
    logscale = 0.0
    for j in 1:L-2
        v₁ = mats[end-j] * v₀
        n = norm(v₁)
        logscale += log(n)
        v₀ = v₁ / n
    end

    ns = ψ[1] * v₀ * exp(logscale)

    return Array(ns, inds(ns)...)
end

number_left (generic function with 1 method)

In [137]:
function simple_circuit(state::DiagonalStateMPS, L::Int, T::Int, p::Float64; cutoff=1e-8, maxdim=200)
    SWAPn1 = decoherence_layer(state, SWAP, p, 1:2:L-1)
    SWAPn2 = decoherence_layer(state, SWAP, p, 2:2:L-1)

    for t in 1:T
        state = apply(SWAPn1, state; cutoff=cutoff, maxdim=maxdim)
        state = apply(SWAPn2, state; cutoff=cutoff, maxdim=maxdim)
        normalize!(state)
        truncate!(state; cutoff=cutoff, maxdim=maxdim)
    end

    return state
end

simple_circuit (generic function with 1 method)

In [185]:
function number_reduce_left(state::DiagonalStateMPS, A::Int)
    state = deepcopy(state)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites)


    dist = ITensor(1.0)
    dist *= ψ[1]
    i1 = only(inds(ψ[1],"Site"))
    for (counter, site) in enumerate(1:A-1)
        i2 = only(inds(ψ[site+1],"Site"))
        i3 = siteind("Qudit", 0; dim=counter+2, conserve_number=true, addtags="Left")

        T = ITensor(dag(i1), dag(i2), i3)
        for a in 1:counter+1, b in 1:2
            c = a + b - 1
            T[i1 => a, i2 => b, i3 => c] = 1.0
        end

        dist *= ψ[site+1]*T
        i1 = i3
    end

    return DiagonalStateMPS(MPS([dist, ψ[A+1:end]...]))
end

function number_reduce_right(state::DiagonalStateMPS, A::Int)
    if A == 0
        return state
    end
    state = deepcopy(state)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites)


    dist = ITensor(1.0)
    dist *= ψ[end]
    i1 = only(inds(ψ[end],"Site"))
    for (counter, site) in enumerate(1:A-1)
        i2 = only(inds(ψ[end-site],"Site"))
        i3 = siteind("Qudit", -1; dim=counter+2, conserve_number=true, addtags="Right")

        T = ITensor(dag(i1), dag(i2), i3)
        for a in 1:counter+1, b in 1:2
            c = a + b - 1
            T[i1 => a, i2 => b, i3 => c] = 1.0
        end

        dist *= ψ[end-site]*T
        i1 = i3
    end

    return DiagonalStateMPS(MPS([ψ[1:end-A]..., dist]))
end

number_reduce_right (generic function with 1 method)

In [139]:
function expval(state::DiagonalStateMPS, M::AbstractMatrix, pos::Int; cutoff=1E-8, maxdim=200, refs=0, conserve_qns=false)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites) - refs
    M_width = Int(log2(size(M)[1]))

    Mψ = apply(op(M, [sites[mod1(pos+i,L)] for i in 0:M_width-1]...), ψ; cutoff=cutoff, maxdim=maxdim)
    
    val = exp(lognorm(Mψ) - lognorm(ψ))
    return val
end

function measure(state::DiagonalStateMPS, M::AbstractMatrix, λ::Float64, pos::Int, m::Bool; cutoff=1E-8, maxdim=200, refs=0)
    ψ = state.mps
    sites = siteinds(ψ)
    L = length(sites) - refs
    M_width = Int(log2(size(M)[1]))

    Π = (I + (-1)^m * λ*M)/(sqrt(2*(1+λ^2)))
    g = op(Π*Π, [sites[mod1(pos+i,L)] for i in 0:M_width-1]...)

    ψ = apply(g, ψ; cutoff=cutoff, maxdim=maxdim)
    
    state = DiagonalStateMPS(ψ)
    truncate!(state; cutoff=cutoff, maxdim=maxdim)
    normalize!(state)

    return state
end

measure (generic function with 9 methods)

In [164]:
L = 200
state = neel_state(DiagonalStateMPS, L; conserve_number=true)

state = simple_circuit(state, L, 1000, 0.1; cutoff=1e-8, maxdim=200)

DiagonalStateMPS(MPS(200))

In [190]:
state2 = number_reduce_right(number_reduce_left(state, 100), 50)

DiagonalStateMPS(MPS(52))

In [191]:
state3 = deepcopy(state2)

DiagonalStateMPS(MPS(52))

In [192]:
state3, _, _ = measure(state3, PauliZ, 1.0, 2:51; cutoff=1e-8, maxdim=200)

(DiagonalStateMPS(MPS(52)), Bool[0, 0, 0, 1, 0, 1, 1, 0, 0, 1  …  0, 0, 0, 0, 1, 0, 0, 0, 1, 0], ComplexF64[-3.929392613872642e-6 + 4.812118087172383e-22im, -0.01959140823109043 + 2.3992555380994004e-18im, -0.03991905036334258 + 4.888673725246948e-18im, -0.060969031562732055 + 7.466552935041376e-18im, -0.04015281436731454 + 4.9173015591689606e-18im, -0.06192119651090994 + 7.583159510646011e-18im, -0.04066503465081108 + 4.9800304522332e-18im, -0.019073502560303258 + 2.335830385900423e-18im, -0.04094301672351347 + 5.014073437788733e-18im, -0.06340346663681624 + 7.764685247162301e-18im  …  -0.03907634232954971 + 4.785471755626928e-18im, -0.0764724415766237 + 9.365173079979517e-18im, -0.11591817566119608 + 1.419588227864844e-17im, -0.15700994068318336 + 1.922817212919763e-17im, -0.20040608475067262 + 2.4542667021956443e-17im, -0.16377133484063355 + 2.005620410046713e-17im, -0.2042757804583642 + 2.5016568068166315e-17im, -0.2459434889894457 + 3.01193906562057e-17im, -0.2903471699529382 + 3.

In [ ]:
number_left(state3.mps)[45:55]

11-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.002194868580225153
 0.03423879190327421
 0.22282199738154757
 0.44599945756459247
 0.24848646271245314
 0.042497062601367964
 0.0037613592565399383

In [143]:
number_left(state2.mps)

3-element Vector{Float64}:
 0.16666857700771967
 0.6666666666561571
 0.1666647563361234

In [132]:
[expval(state2, PauliZ, i) for i in 2:7]

6-element Vector{Number}:
                     1.0000000000000002
                     1.0000000000000002
                     1.0000000000000002
 -1.0000000000000002 + 1.2246467991473535e-16im
                     1.0
                -1.0 + 1.2246467991473532e-16im

In [108]:
ψ = state2.mps

orthogonalize!(ψ, 1)
orthogonalize!(ψ, 3)

maxlinkdim(ψ)

6

In [98]:
siteinds(state2.mps)

4-element Vector{Index{Vector{Pair{QN, Int64}}}}:
 (dim=5|id=277|"Left,Qudit,Site,n=0") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 3: QN("Number",2) => 1
 4: QN("Number",3) => 1
 5: QN("Number",4) => 1
 (dim=2|id=668|"Qubit,Site,n=5") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 (dim=2|id=1|"Qubit,Site,n=6") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 (dim=5|id=961|"Qudit,Right,Site,n=-1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 3: QN("Number",2) => 1
 4: QN("Number",3) => 1
 5: QN("Number",4) => 1

In [97]:
siteinds(state3.mps)

6-element Vector{Index{Vector{Pair{QN, Int64}}}}:
 (dim=2|id=633|"Qubit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 (dim=2|id=598|"Qubit,Site,n=2") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 (dim=2|id=731|"Qubit,Site,n=3") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 (dim=2|id=87|"Qubit,Site,n=4") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 (dim=2|id=395|"Qubit,Site,n=5") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 (dim=6|id=88|"Qudit,Site,n=1") <Out>
 1: QN("Number",0) => 1
 2: QN("Number",1) => 1
 3: QN("Number",2) => 1
 4: QN("Number",3) => 1
 5: QN("Number",4) => 1
 6: QN("Number",5) => 1

In [62]:
lognorm(state2)

2.2109184028007434e-6

In [54]:
truncate!(state; maxdim=2)

DiagonalStateMPS(MPS(10))